# PPAR-alpha (1K7L) Redocking: Smina

PPAR-alpha's standard AutoDock Vina redocking validation failed (RMSD =
2.728 A at exhaustiveness=72; a near-native pose ~0.7-0.9 A was present
but buried at rank 2, not the top-scoring pose -- the same
search-succeeds/scoring-fails pattern already resolved for ACE using
Smina). This notebook tests whether Smina resolves PPAR-alpha's
redocking the same way, using the identical receptor, native ligand,
and box already established for this target.

No result is assumed here -- everything below reports whatever Smina
actually produces.

In [1]:
import os
import subprocess
import pandas as pd
import numpy as np
from pymol import cmd
from rdkit import Chem
from rdkit.Chem import rdFMCS

BIN = "/home/ali/medical plants/docking/Jupyter_Dock-main/bin/"

import sys
sys.path.insert(1, 'utilities/')
from utils import getbox

DOCKING_DIR = "/home/ali/Desktop/Algerian_Lamiaceae_v2/results/docking_results/"
os.chdir(DOCKING_DIR)
print("Working directory:", os.getcwd())

RECEPTOR_PDB = "1K7L_clean_H.pdb"
NATIVE_LIG_MOL2 = "1K7L_native_lig.mol2"

for f in [RECEPTOR_PDB, NATIVE_LIG_MOL2]:
    exists = os.path.exists(f)
    print(f, "exists:", exists)
    if not exists:
        print(f"  -> not found in {DOCKING_DIR}. Run the receptor-prep cells")
        print("     from the original PPAR-alpha notebook first, or update")
        print("     DOCKING_DIR above if this project's files live elsewhere.")


Working directory: /home/ali/Desktop/Algerian_Lamiaceae_v2/results/docking_results
1K7L_clean_H.pdb exists: True
1K7L_native_lig.mol2 exists: True


## Recompute the docking box

Matching the original protocol for this target exactly: no fixed/published
box was established for PPAR-alpha (unlike ACE), so the box is recomputed
via `getbox()` here, identical to how it was originally generated.

In [2]:
cmd.load(filename=RECEPTOR_PDB, format='pdb', object='prot', state=1)
cmd.load(filename=NATIVE_LIG_MOL2, format='mol2', object='natlig', state=1)

center, size = getbox(selection='natlig', extending=5.0, software='vina')
cmd.delete('all')
print(center, '\n', size)


{'center_x': -16.445500373840332, 'center_y': -15.031999588012695, 'center_z': -3.883500039577484} 
 {'size_x': 20.571001052856445, 'size_y': 18.091999053955078, 'size_z': 20.402999997138977}


## RMSD utility (MCS-based)

Same bond-order-agnostic method used throughout this project, since
GW409544's crystal mol2 has had bond-order/aromaticity issues in earlier
attempts (see this target's original redocking-validation cell).

In [3]:
def rmsd_via_mcs(ref, probe):
    mcs = rdFMCS.FindMCS(
        [ref, probe],
        atomCompare=rdFMCS.AtomCompare.CompareElements,
        bondCompare=rdFMCS.BondCompare.CompareAny,
        ringMatchesRingOnly=False,
        matchValences=False,
        timeout=10,
    )
    patt = Chem.MolFromSmarts(mcs.smartsString)
    ref_match = ref.GetSubstructMatch(patt)
    probe_match = probe.GetSubstructMatch(patt)
    if not ref_match or not probe_match:
        return None, 0
    ref_conf = ref.GetConformer()
    probe_conf = probe.GetConformer()
    diffs = []
    for ai, bi in zip(ref_match, probe_match):
        p1 = np.array(ref_conf.GetAtomPosition(ai))
        p2 = np.array(probe_conf.GetAtomPosition(bi))
        diffs.append(np.sum((p1 - p2) ** 2))
    rmsd = round(float(np.sqrt(np.mean(diffs))), 3)
    return rmsd, len(ref_match)

ref_mol_raw = Chem.MolFromMol2File(NATIVE_LIG_MOL2, sanitize=False, removeHs=True)
print("Reference ligand loaded, atom count:", ref_mol_raw.GetNumAtoms())


Reference ligand loaded, atom count: 38


[18:55:23] 1K7L: Warning - no explicit hydrogens in mol2 file but needed for formal charge estimation.


## Smina redocking

Same exhaustiveness (10) as this target's compound-screening protocol,
for direct comparability -- not the raised exhaustiveness (72) that was
needed for Vina's redocking, since we're testing whether a different
scoring function resolves this without needing that adjustment.

In [4]:
SMINA_BIN = BIN + "smina"
assert os.path.exists(SMINA_BIN), f"{SMINA_BIN} does not exist -- check the bin folder."

smina_out = "1K7L_native_lig_smina_out.sdf"

r = subprocess.run([
    SMINA_BIN,
    "-r", RECEPTOR_PDB,
    "-l", NATIVE_LIG_MOL2,
    "-o", smina_out,
    "--center_x", str(center['center_x']), "--center_y", str(center['center_y']), "--center_z", str(center['center_z']),
    "--size_x", str(size['size_x']), "--size_y", str(size['size_y']), "--size_z", str(size['size_z']),
    "--exhaustiveness", "10",
    "--num_modes", "10",
], capture_output=True, text=True)

print("stdout:", r.stdout)
print("stderr:", r.stderr)
print("return code:", r.returncode)


stdout:    _______  _______ _________ _        _______ 
  (  ____ \(       )\__   __/( (    /|(  ___  )
  | (    \/| () () |   ) (   |  \  ( || (   ) |
  | (_____ | || || |   | |   |   \ | || (___) |
  (_____  )| |(_)| |   | |   | (\ \) ||  ___  |
        ) || |   | |   | |   | | \   || (   ) |
  /\____) || )   ( |___) (___| )  \  || )   ( |
  \_______)|/     \|\_______/|/    )_)|/     \|


smina is based off AutoDock Vina. Please cite appropriately.

Weights      Terms
-0.035579    gauss(o=0,_w=0.5,_c=8)
-0.005156    gauss(o=3,_w=2,_c=8)
0.840245     repulsion(o=0,_c=8)
-0.035069    hydrophobic(g=0.5,_b=1.5,_c=8)
-0.587439    non_dir_h_bond(g=-0.7,_b=0,_c=8)
1.923        num_tors_div

Using random seed: -1644330359

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+--

In [5]:
smina_poses = Chem.SDMolSupplier(smina_out, removeHs=True, sanitize=False)

smina_rows = []
for i, pose in enumerate(smina_poses):
    if pose is None:
        smina_rows.append({"Pose": i + 1, "Score_kcal_mol": None, "RMSD_A": None})
        continue
    score = None
    if pose.HasProp("minimizedAffinity"):
        score = float(pose.GetProp("minimizedAffinity"))
    rmsd, n_matched = rmsd_via_mcs(ref_mol_raw, pose)
    print(f"Pose {i+1}: score={score}, matched {n_matched}/{ref_mol_raw.GetNumAtoms()} atoms, RMSD={rmsd}")
    smina_rows.append({"Pose": i + 1, "Score_kcal_mol": score, "RMSD_A": rmsd})

smina_df = pd.DataFrame(smina_rows).sort_values("Score_kcal_mol").reset_index(drop=True)
print()
print("=== Smina results, ranked by score ===")
display(smina_df)

top_smina = smina_df.iloc[0]
print(f"\nTop-scoring pose: score={top_smina['Score_kcal_mol']}, RMSD={top_smina['RMSD_A']} A")
print("PASS" if (top_smina['RMSD_A'] is not None and top_smina['RMSD_A'] < 2.0) else "FAIL")

smina_df.to_csv("PPAR_1K7L_smina_redocking_results.csv", index=False)
print()
print("Saved: PPAR_1K7L_smina_redocking_results.csv")


Pose 1: score=-11.25918, matched 38/38 atoms, RMSD=1.539
Pose 2: score=-10.9617, matched 38/38 atoms, RMSD=8.41
Pose 3: score=-10.84365, matched 38/38 atoms, RMSD=8.328
Pose 4: score=-10.67523, matched 38/38 atoms, RMSD=4.479
Pose 5: score=-10.49534, matched 38/38 atoms, RMSD=9.523
Pose 6: score=-10.26261, matched 38/38 atoms, RMSD=1.851
Pose 7: score=-9.9879, matched 38/38 atoms, RMSD=8.311
Pose 8: score=-9.64233, matched 38/38 atoms, RMSD=8.965
Pose 9: score=-9.3858, matched 38/38 atoms, RMSD=9.564
Pose 10: score=-9.26822, matched 38/38 atoms, RMSD=8.016

=== Smina results, ranked by score ===


[18:55:56] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[18:55:56] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[18:55:56] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[18:55:56] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[18:55:56] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[18:55:56] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[18:55:56] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[18:55:56] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[18:55:56] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

,Pose,Score_kcal_mol,RMSD_A
0,1,-11.25918,1.539
1,2,-10.96170,8.410
2,3,-10.84365,8.328
3,4,-10.67523,4.479
4,5,-10.49534,9.523
5,6,-10.26261,1.851
6,7,-9.98790,8.311
7,8,-9.64233,8.965
8,9,-9.38580,9.564
9,10,-9.26822,8.016



Top-scoring pose: score=-11.25918, RMSD=1.539 A
PASS

Saved: PPAR_1K7L_smina_redocking_results.csv
